In [64]:
pip install lightgbm xgboost catboost optuna


In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from xgboost import XGBRegressor
from catboost import CatBoostRegressor, Pool
import re
import optuna
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit
from scipy.stats import spearmanr

In [4]:
# ==========================================
# COMPUTE SHARPE SCORE
# ==========================================
import numpy as np
import pandas as pd

SOLUTION_NULL_FILLER = -999999

def rank_correlation_sharpe_ratio(merged_df: pd.DataFrame) -> float:
    prediction_cols = [col for col in merged_df.columns if col.startswith('prediction_')]
    target_cols = [col for col in merged_df.columns if col.startswith('target_')]

    def _compute_rank_correlation(row):
        non_null_targets = [col for col in target_cols if not pd.isnull(row[col])]
        matching_predictions = [col for col in prediction_cols if col.replace('prediction', 'target') in non_null_targets]
        if not non_null_targets:
            return np.nan # Gracefully handle rows with no data
        if row[non_null_targets].std(ddof=0) == 0 or row[matching_predictions].std(ddof=0) == 0:
            return 0.0 # Return 0 correlation if there is no variance
        return np.corrcoef(row[matching_predictions].rank(method='average'), row[non_null_targets].rank(method='average'))[0, 1]

    daily_rank_corrs = merged_df.apply(_compute_rank_correlation, axis=1).dropna()
    std_dev = daily_rank_corrs.std(ddof=0)
    if std_dev == 0: return 0.0
    return float(daily_rank_corrs.mean() / std_dev)

def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str) -> float:
    solution = solution.copy()
    submission = submission.copy()
    del solution[row_id_column_name]
    del submission[row_id_column_name]
    
    submission = submission.rename(columns={col: col.replace('target_', 'prediction_') for col in submission.columns})
    solution = solution.replace(SOLUTION_NULL_FILLER, None)
    return rank_correlation_sharpe_ratio(pd.concat([solution, submission], axis='columns'))

In [5]:
# ==========================================
# DATA LOADING
# ==========================================
# Using the specific files provided in your environment
PATH = "C://Users//amogh//OneDrive//Documents//MSBA//Predictive Analytics//mitsui-commodity-prediction-challenge//"
X_train = pd.read_csv(PATH + "X_train_all.csv")
Y_train = pd.read_csv(PATH + "Y_train_all.csv")
X_val = pd.read_csv(PATH + "X_val_all.csv")
Y_val = pd.read_csv(PATH + "Y_val_all.csv")
# Holdout validation set: used only for the reported validation score (cell 7). Not used for Optuna, re-rank, or ES.
X_holdout_val = X_val
Y_holdout_val = Y_val
target_pairs = pd.read_csv(PATH +"target_pairs.csv")
obs_val = pd.read_csv(PATH + "obs_val_all.csv")

# --- Monitor tail (end of train rows): Optuna inputs, re-rank score, early stopping only ---
MONITOR_FRAC = 0.18
_n_mon = max(50, int(len(X_train) * MONITOR_FRAC))
X_fit = X_train.iloc[: -_n_mon].copy()
Y_fit = Y_train.iloc[: -_n_mon].copy()
X_monitor = X_train.iloc[-_n_mon :].copy()
Y_monitor = Y_train.iloc[-_n_mon :].copy()
_monitor_date_ids = np.arange(len(X_monitor), dtype=np.int64)
print(
    f"Train split: fit={len(X_fit)} rows | monitor_tail={len(X_monitor)} | "
    f"holdout_val={len(X_holdout_val)} (report-only score on holdout)"
)

Train split: fit=1271 rows | monitor_tail=278 | holdout_val=392 (report-only score on holdout)


In [7]:
# ==========================================
# FEATURE SELECTION
# ==========================================
import re

def get_pair_specific_features(target_id, target_pairs_df, all_features):
    # 1. Get the row
    row = target_pairs_df[target_pairs_df['target'].astype(str) == str(target_id)]
    if row.empty:
        return []
    
    pair_logic = str(row['pair'].values[0])
    t_lag = int(row['lag'].values[0])
    
    # 2. Use your original working split logic
    # This extracts the base instrument names
    base_instruments = [i.strip() for i in re.split(r'\s*[-+]\s*', pair_logic)]
    
    # 3. Macro Core (The context needed for 0.7 score)
    macro_core = ["FX_USD", "US_Stock_SPY", "US_Stock_VIX", "US_Stock_USO", 
                  "US_Stock_XME", "US_Stock_HYG", "US_Stock_MCHI", "US_Stock_TLT"]
    
    all_needed_bases = list(set(base_instruments + macro_core))
    
    # 4. Improved Selection (Includes your engineered features)
    relevant_cols = []
    for col in all_features:
        for base in all_needed_bases:
            if base in col:
                # ONLY block if the column explicitly mentions a DIFFERENT lag
                # This allows 'base', 'base_ret1', 'base_rmean5', AND 'base_lag1'
                other_lags = [f"_lag{i}" for i in [1, 2, 3, 5, 10] if i != t_lag]
                if not any(bad_lag in col for bad_lag in other_lags):
                    relevant_cols.append(col)
                    break 
                
    return list(set(relevant_cols))


In [5]:
# ==========================================
# HYPERPARAMETER TUNING — LightGBM, XGBoost, CatBoost
# ==========================================

import json
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

import lightgbm as lgb
from xgboost import XGBRegressor
from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_absolute_error

_macro_h = [
    "FX_USD", "US_Stock_SPY", "US_Stock_VIX", "US_Stock_USO",
    "US_Stock_XME", "US_Stock_HYG", "US_Stock_MCHI", "US_Stock_TLT",
]
_feat_h = X_train.columns.tolist()
_sample_targets = list(Y_train.columns[: min(5, len(Y_train.columns))])

N_OPTUNA_TRIALS = 15
TOP_K_RERANK = min(5, N_OPTUNA_TRIALS)
RERANK_SEEDS = [42]


def _xy_for_tune(target_col):
    tf = get_pair_specific_features(target_col, target_pairs, _feat_h)
    tf = list(set(tf + [c for c in _feat_h if any(m in c for m in _macro_h)]))
    if not tf:
        return None, None
    return X_fit[tf], Y_fit[target_col]


# ----- LightGBM -----
def _cv_mean_mae_lgb(X, y, params, n_splits=3, num_boost_round=400):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    out = []
    for tr, va in tscv.split(X):
        dtr = lgb.Dataset(X.iloc[tr], y.iloc[tr])
        dva = lgb.Dataset(X.iloc[va], y.iloc[va], reference=dtr)
        bst = lgb.train(
            params,
            dtr,
            num_boost_round=num_boost_round,
            valid_sets=[dva],
            callbacks=[lgb.early_stopping(30, verbose=False)],
        )
        vs = bst.best_score.get("valid_0", {})
        out.append(float(vs.get("mae", vs.get("l1", next(iter(vs.values()), 0.0)))))
    return float(np.mean(out))


def _optuna_objective_lgb(trial):
    p = {
        "objective": "regression_l1",
        "metric": "mae",
        "boosting_type": "gbdt",
        "verbosity": -1,
        "bagging_freq": 5,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 31, 127),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 0.9),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 0.9),
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-3, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-3, 10.0, log=True),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 20, 100),
    }
    losses = []
    for tcol in _sample_targets:
        Xs, ys = _xy_for_tune(tcol)
        if Xs is None:
            continue
        losses.append(_cv_mean_mae_lgb(Xs, ys, p))
    return float(np.mean(losses)) if losses else 1e9


def _numeric_lgb_from_trial(bp):
    return {
        "learning_rate": float(bp["learning_rate"]),
        "num_leaves": int(bp["num_leaves"]),
        "feature_fraction": float(bp["feature_fraction"]),
        "bagging_fraction": float(bp["bagging_fraction"]),
        "lambda_l1": float(bp["lambda_l1"]),
        "lambda_l2": float(bp["lambda_l2"]),
        "min_data_in_leaf": int(bp["min_data_in_leaf"]),
    }


def _monitor_preds_lgb(lgb_numeric, seeds, all_names):
    lp_base = {
        "objective": "regression_l1",
        "metric": "mae",
        "boosting_type": "gbdt",
        "bagging_freq": 5,
        "verbosity": -1,
        **lgb_numeric,
    }
    pd_out = {}
    for target_col in Y_train.columns:
        target_features = get_pair_specific_features(target_col, target_pairs, all_names)
        target_features = list(
            set(target_features + [c for c in all_names if any(m in c for m in _macro_h)])
        )
        if len(target_features) == 0:
            continue
        X_tr_sub = X_fit[target_features]
        X_mon_sub = X_monitor[target_features]
        y_tr = Y_fit[target_col]
        acc = np.zeros(len(X_mon_sub))
        for s in seeds:
            lp = {**lp_base, "random_state": int(s)}
            dtrain = lgb.Dataset(X_tr_sub, label=y_tr)
            dval = lgb.Dataset(X_mon_sub, label=Y_monitor[target_col], reference=dtrain)
            model = lgb.train(
                lp,
                dtrain,
                num_boost_round=1000,
                valid_sets=[dtrain, dval],
                valid_names=["train", "valid"],
                callbacks=[
                    lgb.early_stopping(stopping_rounds=50),
                    lgb.log_evaluation(period=0),
                ],
            )
            acc += model.predict(X_mon_sub)
        pd_out[target_col] = acc / len(seeds)
    return pd_out


# ----- XGBoost -----
def _cv_mean_mae_xgb(X, y, xgb_hp, n_splits=3, n_estimators=400):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    out = []
    for tr, va in tscv.split(X):
        reg = XGBRegressor(
            objective="reg:absoluteerror",
            tree_method="hist",
            n_estimators=n_estimators,
            early_stopping_rounds=30,
            random_state=42,
            n_jobs=-1,
            **xgb_hp,
        )
        reg.fit(
            X.iloc[tr],
            y.iloc[tr],
            eval_set=[(X.iloc[va], y.iloc[va])],
            verbose=False,
        )
        pred = reg.predict(X.iloc[va])
        out.append(mean_absolute_error(y.iloc[va], pred))
    return float(np.mean(out))


def _optuna_objective_xgb(trial):
    hp = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "subsample": trial.suggest_float("subsample", 0.6, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 0.9),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
    }
    losses = []
    for tcol in _sample_targets:
        Xs, ys = _xy_for_tune(tcol)
        if Xs is None:
            continue
        losses.append(_cv_mean_mae_xgb(Xs, ys, hp))
    return float(np.mean(losses)) if losses else 1e9


def _xgb_hp_from_trial(bp):
    return {
        "learning_rate": float(bp["learning_rate"]),
        "max_depth": int(bp["max_depth"]),
        "subsample": float(bp["subsample"]),
        "colsample_bytree": float(bp["colsample_bytree"]),
        "reg_alpha": float(bp["reg_alpha"]),
        "reg_lambda": float(bp["reg_lambda"]),
        "min_child_weight": int(bp["min_child_weight"]),
    }


def _monitor_preds_xgb(xgb_hp, seeds, all_names):
    pd_out = {}
    for target_col in Y_train.columns:
        target_features = get_pair_specific_features(target_col, target_pairs, all_names)
        target_features = list(
            set(target_features + [c for c in all_names if any(m in c for m in _macro_h)])
        )
        if len(target_features) == 0:
            continue
        X_tr_sub = X_fit[target_features]
        X_mon_sub = X_monitor[target_features]
        y_tr = Y_fit[target_col]
        y_mon = Y_monitor[target_col]
        acc = np.zeros(len(X_mon_sub))
        for s in seeds:
            reg = XGBRegressor(
                objective="reg:absoluteerror",
                tree_method="hist",
                n_estimators=1000,
                early_stopping_rounds=50,
                random_state=int(s),
                n_jobs=-1,
                **xgb_hp,
            )
            reg.fit(X_tr_sub, y_tr, eval_set=[(X_mon_sub, y_mon)], verbose=False)
            acc += reg.predict(X_mon_sub)
        pd_out[target_col] = acc / len(seeds)
    return pd_out


# ----- CatBoost -----
def _cv_mean_mae_cb(X, y, cb_hp, n_splits=3, iterations=400):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    out = []
    for tr, va in tscv.split(X):
        tr_pool = Pool(X.iloc[tr], y.iloc[tr])
        va_pool = Pool(X.iloc[va], y.iloc[va])
        model = CatBoostRegressor(
            iterations=iterations,
            loss_function="MAE",
            early_stopping_rounds=30,
            random_seed=42,
            verbose=False,
            allow_writing_files=False,
            **cb_hp,
        )
        model.fit(tr_pool, eval_set=va_pool, verbose=False)
        pred = model.predict(X.iloc[va])
        out.append(mean_absolute_error(y.iloc[va], pred))
    return float(np.mean(out))


def _optuna_objective_cb(trial):
    hp = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 30.0, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 0.95),
        "rsm": trial.suggest_float("rsm", 0.6, 0.95),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 5, 80),
    }
    losses = []
    for tcol in _sample_targets:
        Xs, ys = _xy_for_tune(tcol)
        if Xs is None:
            continue
        losses.append(_cv_mean_mae_cb(Xs, ys, hp))
    return float(np.mean(losses)) if losses else 1e9


def _cb_hp_from_trial(bp):
    return {
        "learning_rate": float(bp["learning_rate"]),
        "depth": int(bp["depth"]),
        "l2_leaf_reg": float(bp["l2_leaf_reg"]),
        "subsample": float(bp["subsample"]),
        "rsm": float(bp["rsm"]),
        "min_data_in_leaf": int(bp["min_data_in_leaf"]),
    }


def _monitor_preds_cb(cb_hp, seeds, all_names):
    pd_out = {}
    for target_col in Y_train.columns:
        target_features = get_pair_specific_features(target_col, target_pairs, all_names)
        target_features = list(
            set(target_features + [c for c in all_names if any(m in c for m in _macro_h)])
        )
        if len(target_features) == 0:
            continue
        X_tr_sub = X_fit[target_features]
        X_mon_sub = X_monitor[target_features]
        y_tr = Y_fit[target_col]
        y_mon = Y_monitor[target_col]
        acc = np.zeros(len(X_mon_sub))
        for s in seeds:
            model = CatBoostRegressor(
                iterations=1000,
                loss_function="MAE",
                early_stopping_rounds=50,
                random_seed=int(s),
                verbose=False,
                allow_writing_files=False,
                **cb_hp,
            )
            model.fit(
                Pool(X_tr_sub, y_tr),
                eval_set=Pool(X_mon_sub, y_mon),
                verbose=False,
            )
            acc += model.predict(X_mon_sub)
        pd_out[target_col] = acc / len(seeds)
    return pd_out


def _rerank_study(study, monitor_preds_fn, hp_from_trial_fn, label):
    _completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    _completed.sort(key=lambda t: t.value)
    _top = _completed[:TOP_K_RERANK]
    print(
        f"\n[{label}] Re-ranking top {len(_top)} trial(s) on monitor tail "
        f"(RERANK_SEEDS={RERANK_SEEDS})..."
    )
    rows = []
    best_hp = None
    best_score = -np.inf
    for i, tr in enumerate(_top):
        hp = hp_from_trial_fn(tr.params)
        print(f"  [{i + 1}/{len(_top)}] CV_MAE={tr.value:.6f} — monitor preds...")
        pd_mon = monitor_preds_fn(hp, tuple(RERANK_SEEDS), _feat_h)
        sub = pd.DataFrame(pd_mon, index=X_monitor.index)
        sub.insert(0, "date_id", _monitor_date_ids)
        sol = Y_monitor.copy()
        sol.insert(0, "date_id", _monitor_date_ids)
        try:
            vm = float(score(sol, sub, "date_id"))
        except Exception as ex:
            vm = float("nan")
            print(f"    score() failed: {ex}")
        rows.append((vm, tr.value, hp))
        if vm == vm and vm > best_score:
            best_score = vm
            best_hp = hp
    print(f"\n[{label}] Rerank summary (monitor score, higher is better):")
    for vm, mae, hp in sorted(
        rows, key=lambda x: x[0] if x[0] == x[0] else -np.inf, reverse=True
    ):
        print(f"  score={vm:.6f}  CV_MAE={mae:.6f}  {hp}")
    if best_hp is None:
        best_hp = hp_from_trial_fn(study.best_params)
        print(f"[{label}] Rerank invalid; using Optuna-best (MAE).")
    return best_hp, best_score


# ---------- Run LightGBM ----------
print("=" * 60)
print("LightGBM — Optuna (time-series CV MAE, sample targets)")
print("=" * 60)
_study_lgb = optuna.create_study(direction="minimize")
_study_lgb.optimize(_optuna_objective_lgb, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
print("LightGBM Optuna best avg CV MAE:", round(_study_lgb.best_value, 6))
lgb_tuned_numeric, _lgb_rerank_score = _rerank_study(
    _study_lgb, _monitor_preds_lgb, _numeric_lgb_from_trial, "LightGBM"
)

# ---------- Run XGBoost ----------
print("=" * 60)
print("XGBoost — Optuna (time-series CV MAE, sample targets)")
print("=" * 60)
_study_xgb = optuna.create_study(direction="minimize")
_study_xgb.optimize(_optuna_objective_xgb, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
print("XGBoost Optuna best avg CV MAE:", round(_study_xgb.best_value, 6))
xgb_tuned_params, _xgb_rerank_score = _rerank_study(
    _study_xgb, _monitor_preds_xgb, _xgb_hp_from_trial, "XGBoost"
)

# ---------- Run CatBoost ----------
print("=" * 60)
print("CatBoost — Optuna (time-series CV MAE, sample targets)")
print("=" * 60)
_study_cb = optuna.create_study(direction="minimize")
_study_cb.optimize(_optuna_objective_cb, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
print("CatBoost Optuna best avg CV MAE:", round(_study_cb.best_value, 6))
cb_tuned_params, _cb_rerank_score = _rerank_study(
    _study_cb, _monitor_preds_cb, _cb_hp_from_trial, "CatBoost"
)

# ---------- Print final chosen hyperparameters ----------
lgb_full_params = {
    "objective": "regression_l1",
    "metric": "mae",
    "boosting_type": "gbdt",
    "bagging_freq": 5,
    "verbosity": -1,
    **lgb_tuned_numeric,
}
xgb_full_params = {
    "objective": "reg:absoluteerror",
    "tree_method": "hist",
    **xgb_tuned_params,
}
cb_full_params = dict(cb_tuned_params)

print("\n" + "=" * 60)
print("FINAL CHOSEN HYPERPARAMETERS (after monitor-tail rerank)")
print("=" * 60)
print("\n--- LightGBM ---")
print(json.dumps(lgb_full_params, indent=2))
print("\n--- XGBoost (sklearn API; n_estimators set at train time) ---")
print(json.dumps(xgb_full_params, indent=2))
print("\n--- CatBoost (iterations set at train time) ---")
print(json.dumps(cb_full_params, indent=2))
print(
    "\nBest monitor-tail score during rerank | LightGBM:",
    round(_lgb_rerank_score, 6),
    "| XGBoost:",
    round(_xgb_rerank_score, 6),
    "| CatBoost:",
    round(_cb_rerank_score, 6),
)


LightGBM — Optuna (time-series CV MAE, sample targets)


  0%|          | 0/15 [00:00<?, ?it/s]

LightGBM Optuna best avg CV MAE: 0.010526

[LightGBM] Re-ranking top 5 trial(s) on monitor tail (RERANK_SEEDS=[42])...
  [1/5] CV_MAE=0.010526 — monitor preds...
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[10]	train's l1: 0.00798873	valid's l1: 0.00589571
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[2]	train's l1: 0.0124772	valid's l1: 0.00887073
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[53]	train's l1: 0.00956766	valid's l1: 0.00983504
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	train's l1: 0.0115553	valid's l1: 0.00978045
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	train's l1: 0.00964894	valid's l1: 0.00866251
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[2]	train's 

  0%|          | 0/15 [00:00<?, ?it/s]

XGBoost Optuna best avg CV MAE: 0.010534

[XGBoost] Re-ranking top 5 trial(s) on monitor tail (RERANK_SEEDS=[42])...
  [1/5] CV_MAE=0.010534 — monitor preds...
  [2/5] CV_MAE=0.010534 — monitor preds...
  [3/5] CV_MAE=0.010534 — monitor preds...
  [4/5] CV_MAE=0.010536 — monitor preds...
  [5/5] CV_MAE=0.010536 — monitor preds...

[XGBoost] Rerank summary (monitor score, higher is better):
  score=0.627833  CV_MAE=0.010534  {'learning_rate': 0.011760188143916252, 'max_depth': 11, 'subsample': 0.6740498075439663, 'colsample_bytree': 0.6274322213703454, 'reg_alpha': 0.6490187056934515, 'reg_lambda': 3.8182015265198737, 'min_child_weight': 5}
  score=0.557351  CV_MAE=0.010536  {'learning_rate': 0.013887480577102738, 'max_depth': 5, 'subsample': 0.7384977217460675, 'colsample_bytree': 0.8245842997898044, 'reg_alpha': 0.6982754185603812, 'reg_lambda': 0.8564428367992218, 'min_child_weight': 6}
  score=0.528670  CV_MAE=0.010534  {'learning_rate': 0.011043052814283853, 'max_depth': 4, 'subsam

  0%|          | 0/15 [00:00<?, ?it/s]

CatBoost Optuna best avg CV MAE: 0.010529

[CatBoost] Re-ranking top 5 trial(s) on monitor tail (RERANK_SEEDS=[42])...
  [1/5] CV_MAE=0.010529 — monitor preds...
  [2/5] CV_MAE=0.010532 — monitor preds...
  [3/5] CV_MAE=0.010536 — monitor preds...
  [4/5] CV_MAE=0.010536 — monitor preds...
  [5/5] CV_MAE=0.010536 — monitor preds...

[CatBoost] Rerank summary (monitor score, higher is better):
  score=0.580061  CV_MAE=0.010536  {'learning_rate': 0.011005367809458285, 'depth': 9, 'l2_leaf_reg': 3.613728617871899, 'subsample': 0.6356190063209927, 'rsm': 0.8574039697263778, 'min_data_in_leaf': 77}
  score=0.558192  CV_MAE=0.010536  {'learning_rate': 0.015385836082863038, 'depth': 4, 'l2_leaf_reg': 6.250824345573671, 'subsample': 0.6831070811732999, 'rsm': 0.7821786531222579, 'min_data_in_leaf': 43}
  score=0.520541  CV_MAE=0.010536  {'learning_rate': 0.020606544897157134, 'depth': 7, 'l2_leaf_reg': 29.24763786806311, 'subsample': 0.7497164376208174, 'rsm': 0.6906553684804597, 'min_data_in_

In [6]:
# ==========================================
# TRAINING — LightGBM, XGBoost, CatBoost
# ==========================================

import numpy as np
import lightgbm as lgb
from xgboost import XGBRegressor
from catboost import CatBoostRegressor, Pool

macro_core = [
    "FX_USD", "US_Stock_SPY", "US_Stock_VIX", "US_Stock_USO",
    "US_Stock_XME", "US_Stock_HYG", "US_Stock_MCHI", "US_Stock_TLT",
]

all_feature_names = X_train.columns.tolist()
preds_dict_lgb = {}
preds_dict_xgb = {}
preds_dict_cb = {}

seeds = [42, 2024, 88]

# ----- LightGBM -----
_t_lgb = globals().get("lgb_tuned_numeric")
if _t_lgb is None:
    _t_lgb = {
        "learning_rate": 0.015,
        "num_leaves": 31,
        "feature_fraction": 0.7,
        "bagging_fraction": 0.7,
        "lambda_l1": 1.0,
        "lambda_l2": 1.0,
    }
    print("Note: run tuning cell first for LightGBM tuned params.")

lgb_params = {
    "objective": "regression_l1",
    "metric": "mae",
    "boosting_type": "gbdt",
    "bagging_freq": 5,
    "verbosity": -1,
    **_t_lgb,
}

print("=" * 60)
print("LightGBM — training (ES on monitor; refit full train; predict holdout)")
print("=" * 60)
for target_col in Y_train.columns:
    target_features = get_pair_specific_features(target_col, target_pairs, all_feature_names)
    target_features = list(
        set(target_features + [c for c in all_feature_names if any(m in c for m in macro_core)])
    )
    if len(target_features) == 0:
        print(f"Warning: No features for {target_col}. Skipping...")
        continue
    X_tr_sub_fit = X_fit[target_features]
    X_monitor_sub = X_monitor[target_features]
    X_ho_sub = X_holdout_val[target_features]
    y_train_clean = Y_train[target_col]
    y_fit = Y_fit[target_col]
    y_monitor = Y_monitor[target_col]
    acc = np.zeros(len(X_ho_sub))
    for s in seeds:
        lgb_params["random_state"] = s
        dtrain = lgb.Dataset(X_tr_sub_fit, label=y_fit)
        dval = lgb.Dataset(X_monitor_sub, label=y_monitor, reference=dtrain)
        model = lgb.train(
            lgb_params,
            dtrain,
            num_boost_round=1000,
            valid_sets=[dtrain, dval],
            valid_names=["train", "valid"],
            callbacks=[
                lgb.early_stopping(stopping_rounds=50),
                lgb.log_evaluation(period=0),
            ],
        )
        bi = model.best_iteration
        if bi is None or bi < 1:
            bi = 200
        dfull = lgb.Dataset(X_train[target_features], label=y_train_clean)
        model_full = lgb.train(
            lgb_params,
            dfull,
            num_boost_round=int(bi),
            callbacks=[lgb.log_evaluation(period=0)],
        )
        acc += model_full.predict(X_ho_sub)
    preds_dict_lgb[target_col] = acc / len(seeds)
    print(f"LightGBM OK {target_col} | Features: {len(target_features)}")

# ----- XGBoost -----
_xhp = globals().get("xgb_tuned_params")
if _xhp is None:
    _xhp = {
        "learning_rate": 0.05,
        "max_depth": 6,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.1,
        "reg_lambda": 1.0,
        "min_child_weight": 5,
    }
    print("Note: run tuning cell first for XGBoost tuned params.")

print("=" * 60)
print("XGBoost — training (ES on monitor; refit full train; predict holdout)")
print("=" * 60)
for target_col in Y_train.columns:
    target_features = get_pair_specific_features(target_col, target_pairs, all_feature_names)
    target_features = list(
        set(target_features + [c for c in all_feature_names if any(m in c for m in macro_core)])
    )
    if len(target_features) == 0:
        continue
    X_tr_sub_fit = X_fit[target_features]
    X_monitor_sub = X_monitor[target_features]
    X_ho_sub = X_holdout_val[target_features]
    y_train_clean = Y_train[target_col]
    y_fit = Y_fit[target_col]
    y_monitor = Y_monitor[target_col]
    acc = np.zeros(len(X_ho_sub))
    for s in seeds:
        reg = XGBRegressor(
            objective="reg:absoluteerror",
            tree_method="hist",
            n_estimators=1000,
            early_stopping_rounds=50,
            random_state=int(s),
            n_jobs=-1,
            **_xhp,
        )
        reg.fit(
            X_tr_sub_fit,
            y_fit,
            eval_set=[(X_monitor_sub, y_monitor)],
            verbose=False,
        )
        bi = getattr(reg, "best_iteration", None)
        if bi is None:
            bi = 199
        bi = int(bi)
        reg_full = XGBRegressor(
            objective="reg:absoluteerror",
            tree_method="hist",
            n_estimators=bi + 1,
            random_state=int(s),
            n_jobs=-1,
            **_xhp,
        )
        reg_full.fit(X_train[target_features], y_train_clean, verbose=False)
        acc += reg_full.predict(X_ho_sub)
    preds_dict_xgb[target_col] = acc / len(seeds)
    print(f"XGBoost OK {target_col} | Features: {len(target_features)}")

# ----- CatBoost -----
_chp = globals().get("cb_tuned_params")
if _chp is None:
    _chp = {
        "learning_rate": 0.05,
        "depth": 6,
        "l2_leaf_reg": 3.0,
        "subsample": 0.8,
        "rsm": 0.8,
        "min_data_in_leaf": 20,
    }
    print("Note: run tuning cell first for CatBoost tuned params.")

print("=" * 60)
print("CatBoost — training (ES on monitor; refit full train; predict holdout)")
print("=" * 60)
for target_col in Y_train.columns:
    target_features = get_pair_specific_features(target_col, target_pairs, all_feature_names)
    target_features = list(
        set(target_features + [c for c in all_feature_names if any(m in c for m in macro_core)])
    )
    if len(target_features) == 0:
        continue
    X_tr_sub_fit = X_fit[target_features]
    X_monitor_sub = X_monitor[target_features]
    X_ho_sub = X_holdout_val[target_features]
    y_train_clean = Y_train[target_col]
    y_fit = Y_fit[target_col]
    y_monitor = Y_monitor[target_col]
    acc = np.zeros(len(X_ho_sub))
    for s in seeds:
        model = CatBoostRegressor(
            iterations=1000,
            loss_function="MAE",
            early_stopping_rounds=50,
            random_seed=int(s),
            verbose=False,
            allow_writing_files=False,
            **_chp,
        )
        model.fit(
            Pool(X_tr_sub_fit, y_fit),
            eval_set=Pool(X_monitor_sub, y_monitor),
            verbose=False,
        )
        bi = model.get_best_iteration()
        if bi is None or bi < 0:
            bi = 199
        bi = int(bi)
        model_full = CatBoostRegressor(
            iterations=bi + 1,
            loss_function="MAE",
            random_seed=int(s),
            verbose=False,
            allow_writing_files=False,
            **_chp,
        )
        model_full.fit(Pool(X_train[target_features], y_train_clean), verbose=False)
        acc += model_full.predict(X_ho_sub)
    preds_dict_cb[target_col] = acc / len(seeds)
    print(f"CatBoost OK {target_col} | Features: {len(target_features)}")

# Backward compatibility for cells that still expect preds_dict
preds_dict = preds_dict_lgb

print("\n--- Training complete (LightGBM, XGBoost, CatBoost) ---")


LightGBM — training (ES on monitor; refit full train; predict holdout)
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	train's l1: 0.00802043	valid's l1: 0.00589576
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[21]	train's l1: 0.00769183	valid's l1: 0.00588081
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	train's l1: 0.00802633	valid's l1: 0.00589934
LightGBM OK target_0 | Features: 69
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	train's l1: 0.0124685	valid's l1: 0.00888578
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[2]	train's l1: 0.0124391	valid's l1: 0.0088847
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[3]	train's l1: 0.0124206	valid's l1: 0.00887101
LightGBM OK target_1

In [7]:
# ==========================================
# HOLDOUT VALIDATION — all trained backends
# ==========================================
full_dates = pd.read_csv(PATH + "train.csv", usecols=["date_id"])
val_date_ids = full_dates["date_id"].values[-len(X_holdout_val) :]

print("\n" + "=" * 60)
print("HOLDOUT VALIDATION (report-only; X_val_all / Y_val_all)")
print("=" * 60)

holdout_scores = {}

for name, pdct in [
    ("LightGBM", preds_dict_lgb),
    ("XGBoost", preds_dict_xgb),
    ("CatBoost", preds_dict_cb),
]:
    submission_val = pd.DataFrame(pdct, index=X_holdout_val.index)
    submission_val.insert(0, "date_id", val_date_ids)
    solution_val = Y_holdout_val.copy()
    solution_val.insert(0, "date_id", val_date_ids)
    try:
        sh = float(score(solution_val, submission_val, "date_id"))
        holdout_scores[name] = sh
        print(f"{name} — Holdout validation Sharpe: {sh:.4f}")
    except Exception as e:
        holdout_scores[name] = float("nan")
        print(f"{name} — Error: {e}")

# Default downstream cells use LightGBM submission
submission_val = pd.DataFrame(preds_dict_lgb, index=X_holdout_val.index)
submission_val.insert(0, "date_id", val_date_ids)
solution_val = Y_holdout_val.copy()
solution_val.insert(0, "date_id", val_date_ids)
val_sharpe = holdout_scores.get("LightGBM", float("nan"))



HOLDOUT VALIDATION (report-only; X_val_all / Y_val_all)
LightGBM — Holdout validation Sharpe: 0.1189
XGBoost — Holdout validation Sharpe: 0.0683
CatBoost — Holdout validation Sharpe: 0.1478
